<a href="https://colab.research.google.com/github/vaishnavipoojary2202/ecommerce-customer-behavior-analytics/blob/main/01_data_exploration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# E-Commerce Customer Behavior Analytics — Data Exploration

This notebook is the first stage of the project. We will inspect the October 2019 e-commerce behavior dataset before building the cleaning, analytics, RFM, clustering, and dashboard pipeline.

**Working file:** `2019-Oct.csv`  
**Raw data is not stored in GitHub because of its large size.

In [1]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip -q install pyspark

In [3]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("ECommerceCustomerBehaviorAnalytics").getOrCreate()
print("Spark version:", spark.version)

Spark version: 4.0.4


In [6]:
!ls "/content/drive/MyDrive"

 1000074304.pdf
 17603273621015784220519617026486.jpg
 1760327572463100964819142675928.jpg
 17603277056134962343981861500829.jpg
 17603278282973531728527865690285.jpg
 17603279750808260018271758383665.jpg
 17603283134158855682480205284201.jpg
 1760328638038370163834356956205.jpg
 17603286666497346057828898515117.jpg
 17603291085546384599301330846389.jpg
 17603293218375305079538193289053.jpg
'42(6) (1).gdoc'
'42(6).docx'
'42(6).gdoc'
'42(6).pdf'
 AI-2.gdoc
 Attendance.gsheet
'bank_star_schema (2).svg'
 Bda-5.gdoc
 BDA-EXP2.gdoc
 booths.gdoc
 booths.txt
 ccl03.docx
'CCL8 (1).docx'
 CCL8.docx
 ccl-9.gdoc
 cgl5.pdf_20241009_111606_0000.gdoc
 cgl5.pdf_20241009_111606_0000.pdf
 Classroom
 CLIO
'CN_Exp 1.gdoc'
 COA.gdoc
'Colab Notebooks'
'Computer Network'
'Copy of CSI Mosaic registeration  driver wala .gsheet'
'Copy of Mid Term Evaluation PPT.pptx'
'Copy of MOSAIC 2025 Registration Form.xlsx'
'crypto 6.gdoc'
 Dashboard.gsheet
 DBMS_3VAISH.docx
 Display-Session_Plan.gdoc
 dovetail.gdoc
'Dovet

In [7]:
!ls "/content/drive/MyDrive/BDA"

 Datasets  'Research Papers'


In [8]:
!ls "/content/drive/MyDrive/BDA/Datasets"

2019-Oct.csv


In [10]:
FILE_PATH = "/content/drive/MyDrive/BDA/Datasets/2019-Oct.csv"

df = spark.read.csv(FILE_PATH, header=True, inferSchema=True)
print("Data loaded successfully.")

Data loaded successfully.


In [11]:
print("Rows:", df.count())
print("Columns:", len(df.columns))
print("Column names:", df.columns)

Rows: 42448764
Columns: 9
Column names: ['event_time', 'event_type', 'product_id', 'category_id', 'category_code', 'brand', 'price', 'user_id', 'user_session']


In [12]:
df.printSchema()

root
 |-- event_time: timestamp (nullable = true)
 |-- event_type: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- category_id: long (nullable = true)
 |-- category_code: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- price: double (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- user_session: string (nullable = true)



In [13]:
df.show(10, truncate=False)

+-------------------+----------+----------+-------------------+-----------------------------------+--------+-------+---------+------------------------------------+
|event_time         |event_type|product_id|category_id        |category_code                      |brand   |price  |user_id  |user_session                        |
+-------------------+----------+----------+-------------------+-----------------------------------+--------+-------+---------+------------------------------------+
|2019-10-01 00:00:00|view      |44600062  |2103807459595387724|NULL                               |shiseido|35.79  |541312140|72d76fde-8bb3-4e00-8c23-a032dfed738c|
|2019-10-01 00:00:00|view      |3900821   |2053013552326770905|appliances.environment.water_heater|aqua    |33.2   |554748717|9333dfbd-b87a-4708-9857-6336556b0fcc|
|2019-10-01 00:00:01|view      |17200506  |2053013559792632471|furniture.living_room.sofa         |NULL    |543.1  |519107250|566511c2-e2e3-422b-b695-cf8e6e792ca8|
|2019-10-01 00:0

## Event Type Analysis

In [14]:
df.groupBy("event_type").count().orderBy("count", ascending=False).show()

+----------+--------+
|event_type|   count|
+----------+--------+
|      view|40779399|
|      cart|  926516|
|  purchase|  742849|
+----------+--------+



In [15]:
print("Unique users:", df.select("user_id").distinct().count())
print("Unique products:", df.select("product_id").distinct().count())
print("Unique sessions:", df.select("user_session").distinct().count())

Unique users: 3022290
Unique products: 166794
Unique sessions: 9244422


## Missing Values

In [16]:
from pyspark.sql.functions import col, sum as spark_sum

missing_values = df.select([spark_sum(col(c).isNull().cast("int")).alias(c) for c in df.columns])
missing_values.show(vertical=True)

-RECORD 0-----------------
 event_time    | 0        
 event_type    | 0        
 product_id    | 0        
 category_id   | 0        
 category_code | 13515609 
 brand         | 6113008  
 price         | 0        
 user_id       | 0        
 user_session  | 2        



In [18]:
from pyspark.sql.functions import col, sum as spark_sum, round

total_rows = df.count()

missing_percentage = df.select([
    round(
        (spark_sum(col(c).isNull().cast("int")) / total_rows) * 100,
        2
    ).alias(c)
    for c in df.columns
])

missing_percentage.show(vertical=True)

-RECORD 0--------------
 event_time    | 0.0   
 event_type    | 0.0   
 product_id    | 0.0   
 category_id   | 0.0   
 category_code | 31.84 
 brand         | 14.4  
 price         | 0.0   
 user_id       | 0.0   
 user_session  | 0.0   



## Purchase Data

In [20]:
purchase_df = df.filter(col("event_type") == "purchase")
print("Purchase events:", purchase_df.count())
print("Unique purchasing users:", purchase_df.select("user_id").distinct().count())
print("Unique purchase sessions:", purchase_df.select("user_session").distinct().count())

Purchase events: 742849
Unique purchasing users: 347118
Unique purchase sessions: 629560


In [21]:
purchase_df.show(10, truncate=False)

+-------------------+----------+----------+-------------------+---------------------------------+-------+------+---------+------------------------------------+
|event_time         |event_type|product_id|category_id        |category_code                    |brand  |price |user_id  |user_session                        |
+-------------------+----------+----------+-------------------+---------------------------------+-------+------+---------+------------------------------------+
|2019-10-01 00:02:14|purchase  |1004856   |2053013555631882655|electronics.smartphone           |samsung|130.76|543272936|8187d148-3c41-46d4-b0c0-9c08cd9dc564|
|2019-10-01 00:04:37|purchase  |1002532   |2053013555631882655|electronics.smartphone           |apple  |642.69|551377651|3c80f0d6-e9ec-4181-8c5c-837a30be2d68|
|2019-10-01 00:06:02|purchase  |5100816   |2053013553375346967|NULL                             |xiaomi |29.51 |514591159|0e5dfc4b-2a55-43e6-8c05-97e1f07fbb56|
|2019-10-01 00:07:07|purchase  |13800054

In [22]:
from pyspark.sql.functions import min, max

purchase_df.select(
    min("event_time").alias("first_purchase"),
    max("event_time").alias("last_purchase")
).show(truncate=False)

+-------------------+-------------------+
|first_purchase     |last_purchase      |
+-------------------+-------------------+
|2019-10-01 00:02:14|2019-10-31 23:59:16|
+-------------------+-------------------+



## Date Range

In [23]:
from pyspark.sql.functions import min as spark_min, max as spark_max

df.select(spark_min("event_time").alias("start_time"), spark_max("event_time").alias("end_time")).show()

+-------------------+-------------------+
|         start_time|           end_time|
+-------------------+-------------------+
|2019-10-01 00:00:00|2019-10-31 23:59:59|
+-------------------+-------------------+



In [24]:
from pyspark.sql.functions import (
    min, max, sum as spark_sum, count
)

session_purchase_df = purchase_df.groupBy(
    "user_id",
    "user_session"
).agg(
    min("event_time").alias("session_start"),
    max("event_time").alias("session_end"),
    count("*").alias("items_purchased"),
    spark_sum("price").alias("session_value")
)

session_purchase_df.show(10, truncate=False)

+---------+------------------------------------+-------------------+-------------------+---------------+-------------+
|user_id  |user_session                        |session_start      |session_end        |items_purchased|session_value|
+---------+------------------------------------+-------------------+-------------------+---------------+-------------+
|552439521|80ccabf1-39b6-421f-b452-4bf9e37805d6|2019-10-01 02:27:17|2019-10-01 02:27:17|1              |185.71       |
|555463617|d312a312-0f7e-41b8-a972-d45cead93b3f|2019-10-01 02:35:34|2019-10-01 02:35:34|1              |189.71       |
|514104941|20dd9374-8cc6-4375-9cb7-60be3e0f24e1|2019-10-01 02:44:43|2019-10-01 02:44:43|1              |153.04       |
|549525040|2764f489-f2c1-4836-8ada-6b0e57897e49|2019-10-01 02:47:11|2019-10-01 02:47:11|1              |68.01        |
|532361434|17b959ab-b343-42c1-8e6d-dd7a4c8a9bd9|2019-10-01 02:55:26|2019-10-01 02:55:26|1              |134.42       |
|555394812|f5ca4a9f-89dd-4c74-8e2f-0588abeff975|

In [25]:
print("Session-level transactions:", session_purchase_df.count())
print(
    "Unique customers:",
    session_purchase_df.select("user_id").distinct().count()
)

Session-level transactions: 629560
Unique customers: 347118


In [26]:
from pyspark.sql.functions import (
    sum as spark_sum,
    min,
    max,
    avg
)

session_purchase_df.select(
    spark_sum(col("session_value").isNull().cast("int")).alias("null_session_values"),
    min("session_value").alias("minimum_session_value"),
    max("session_value").alias("maximum_session_value"),
    avg("session_value").alias("average_session_value")
).show()

+-------------------+---------------------+---------------------+---------------------+
|null_session_values|minimum_session_value|maximum_session_value|average_session_value|
+-------------------+---------------------+---------------------+---------------------+
|                  0|                 0.77|             37635.55|    365.2670154869866|
+-------------------+---------------------+---------------------+---------------------+



In [27]:
session_purchase_df.select(
    spark_sum(col("items_purchased").isNull().cast("int")).alias("null_basket_sizes"),
    min("items_purchased").alias("minimum_items"),
    max("items_purchased").alias("maximum_items"),
    avg("items_purchased").alias("average_items")
).show()

+-----------------+-------------+-------------+------------------+
|null_basket_sizes|minimum_items|maximum_items|     average_items|
+-----------------+-------------+-------------+------------------+
|                0|            1|           48|1.1799494885316728|
+-----------------+-------------+-------------+------------------+



In [28]:
percentiles = session_purchase_df.approxQuantile(
    ["session_value", "items_purchased"],
    [0.25, 0.50, 0.75, 0.90, 0.95, 0.99],
    0.01
)

print("Percentiles:")
print("25th percentile:", percentiles[0][0], "|", percentiles[1][0])
print("50th percentile:", percentiles[0][1], "|", percentiles[1][1])
print("75th percentile:", percentiles[0][2], "|", percentiles[1][2])
print("90th percentile:", percentiles[0][3], "|", percentiles[1][3])
print("95th percentile:", percentiles[0][4], "|", percentiles[1][4])
print("99th percentile:", percentiles[0][5], "|", percentiles[1][5])

Percentiles:
25th percentile: 87.78 | 1.0
50th percentile: 190.92 | 1.0
75th percentile: 422.85 | 1.0
90th percentile: 946.97 | 2.0
95th percentile: 1266.87 | 2.0
99th percentile: 37635.55 | 48.0


In [29]:
purchase_df.select(
    spark_sum(col("price").isNull().cast("int")).alias("null_prices"),
    spark_sum((col("price") <= 0).cast("int")).alias("non_positive_prices"),
    min("price").alias("minimum_price"),
    max("price").alias("maximum_price"),
    avg("price").alias("average_price")
).show()

+-----------+-------------------+-------------+-------------+------------------+
|null_prices|non_positive_prices|minimum_price|maximum_price|     average_price|
+-----------+-------------------+-------------+-------------+------------------+
|          0|                  0|         0.77|      2574.07|309.56156940373995|
+-----------+-------------------+-------------+-------------+------------------+



In [30]:
from pyspark.sql.functions import (
    count,
    sum as spark_sum,
    avg,
    max as spark_max,
    datediff,
    lit
)

reference_date = "2019-11-01"

customer_rfm = session_purchase_df.groupBy(
    "user_id"
).agg(
    datediff(
        lit(reference_date),
        spark_max("session_end").cast("date")
    ).alias("recency"),

    count("*").alias("frequency"),

    spark_sum("session_value").alias("monetary"),

    avg("session_value").alias("avg_transaction_value"),

    avg("items_purchased").alias("avg_basket_size")
)

customer_rfm.show(10, truncate=False)

+---------+-------+---------+------------------+---------------------+------------------+
|user_id  |recency|frequency|monetary          |avg_transaction_value|avg_basket_size   |
+---------+-------+---------+------------------+---------------------+------------------+
|540805815|25     |5        |1287.01           |257.402              |1.2               |
|514440503|31     |1        |180.16            |180.16               |1.0               |
|546837996|31     |1        |278.24            |278.24               |1.0               |
|514734316|31     |1        |64.32             |64.32                |1.0               |
|545546853|19     |3        |3029.4900000000002|1009.83              |1.6666666666666667|
|524340312|15     |4        |336.5             |84.125               |1.0               |
|512946423|11     |6        |5272.0            |878.6666666666666    |1.0               |
|544161168|31     |1        |737.67            |737.67               |1.0               |
|527993733

In [31]:
customer_rfm.select(
    spark_sum(col("recency").isNull().cast("int")).alias("null_recency"),
    spark_sum(col("frequency").isNull().cast("int")).alias("null_frequency"),
    spark_sum(col("monetary").isNull().cast("int")).alias("null_monetary"),
    spark_sum(col("avg_transaction_value").isNull().cast("int")).alias("null_atv"),
    spark_sum(col("avg_basket_size").isNull().cast("int")).alias("null_basket")
).show()

+------------+--------------+-------------+--------+-----------+
|null_recency|null_frequency|null_monetary|null_atv|null_basket|
+------------+--------------+-------------+--------+-----------+
|           0|             0|            0|       0|          0|
+------------+--------------+-------------+--------+-----------+



In [32]:
customer_rfm.select(
    min("recency").alias("min_recency"),
    max("recency").alias("max_recency"),
    min("frequency").alias("min_frequency"),
    max("frequency").alias("max_frequency"),
    min("monetary").alias("min_monetary"),
    max("monetary").alias("max_monetary")
).show()

+-----------+-----------+-------------+-------------+------------+-----------------+
|min_recency|max_recency|min_frequency|max_frequency|min_monetary|     max_monetary|
+-----------+-----------+-------------+-------------+------------+-----------------+
|          1|         31|            1|          152|        0.88|265569.5199999999|
+-----------+-----------+-------------+-------------+------------+-----------------+



In [33]:
from pyspark.sql.functions import count

customer_session_counts = session_purchase_df.groupBy(
    "user_id"
).agg(
    count("*").alias("purchase_sessions")
)

customer_session_counts.groupBy(
    "purchase_sessions"
).count().orderBy("purchase_sessions").show(20)

+-----------------+------+
|purchase_sessions| count|
+-----------------+------+
|                1|235461|
|                2| 60952|
|                3| 21918|
|                4| 10338|
|                5|  5428|
|                6|  3383|
|                7|  2238|
|                8|  1538|
|                9|  1110|
|               10|   807|
|               11|   649|
|               12|   497|
|               13|   444|
|               14|   347|
|               15|   247|
|               16|   218|
|               17|   192|
|               18|   173|
|               19|   135|
|               20|   123|
+-----------------+------+
only showing top 20 rows


In [35]:
from pyspark.sql.window import Window
from pyspark.sql.functions import (
    row_number,
    col,
    avg,
    sum as spark_sum,
    pow,
    when
)

# 1. Number each customer's purchase sessions chronologically
session_window = Window.partitionBy("user_id").orderBy("session_start")

session_ordered = session_purchase_df.withColumn(
    "session_number",
    row_number().over(session_window)
)

# 2. Calculate customer-level means
customer_means = session_ordered.groupBy("user_id").agg(
    avg("session_number").alias("mean_session_number"),
    avg("session_value").alias("mean_session_value"),
    spark_sum(col("session_number") * 0 + 1).alias("session_count")
)

# 3. Join customer means back to session data
session_with_means = session_ordered.join(
    customer_means,
    on="user_id",
    how="left"
)

# 4. Calculate numerator and denominator
amount_slope_components = session_with_means.groupBy("user_id").agg(
    spark_sum(
        (col("session_number") - col("mean_session_number")) *
        (col("session_value") - col("mean_session_value"))
    ).alias("numerator"),

    spark_sum(
        pow(
            col("session_number") - col("mean_session_number"),
            2
        )
    ).alias("denominator"),

    spark_max("session_count").alias("session_count")
)

# 5. Calculate slope only when at least 2 sessions exist
amount_slope_df = amount_slope_components.withColumn(
    "amount_slope",
    when(
        col("session_count") >= 2,
        col("numerator") / col("denominator")
    ).otherwise(0.0)
).select(
    "user_id",
    "amount_slope"
)

amount_slope_df.show(10, truncate=False)

+---------+------------------+
|user_id  |amount_slope      |
+---------+------------------+
|264649825|135.76            |
|303160429|0.0               |
|340041246|-17.80000000000001|
|387300134|0.0               |
|396222093|0.0               |
|403013066|0.0               |
|413782099|-12.86999999999999|
|415153398|0.0               |
|418115333|0.0               |
|419697235|0.0               |
+---------+------------------+
only showing top 10 rows


In [36]:
from pyspark.sql.functions import (
    col,
    count,
    sum as spark_sum,
    min as spark_min,
    max as spark_max,
    isnan
)

amount_slope_df.select(
    count("*").alias("customers"),
    spark_sum(col("amount_slope").isNull().cast("int")).alias("null_slopes"),
    spark_sum(isnan("amount_slope").cast("int")).alias("nan_slopes"),
    spark_min("amount_slope").alias("min_slope"),
    spark_max("amount_slope").alias("max_slope")
).show(truncate=False)

+---------+-----------+----------+---------+-----------------+
|customers|null_slopes|nan_slopes|min_slope|max_slope        |
+---------+-----------+----------+---------+-----------------+
|347118   |0          |0         |-10476.18|8751.469999999998|
+---------+-----------+----------+---------+-----------------+



In [37]:
from pyspark.sql.functions import (
    datediff,
    lit,
    floor,
    count,
    col
)

weekly_purchase_df = session_purchase_df.withColumn(
    "days_since_start",
    datediff(
        col("session_start").cast("date"),
        lit("2019-10-01").cast("date")
    )
).withColumn(
    "week_number",
    floor(col("days_since_start") / 7) + 1
)

weekly_purchase_counts = weekly_purchase_df.groupBy(
    "user_id",
    "week_number"
).agg(
    count("*").alias("weekly_purchase_count")
)

weekly_purchase_counts.show(10)

+---------+-----------+---------------------+
|  user_id|week_number|weekly_purchase_count|
+---------+-----------+---------------------+
|514915945|          1|                    1|
|543614983|          1|                    2|
|545196090|          1|                    1|
|555697835|          1|                    1|
|539319122|          1|                    1|
|523060253|          1|                    1|
|543011385|          1|                    1|
|522702221|          1|                    2|
|513275232|          1|                    1|
|517342181|          1|                    1|
+---------+-----------+---------------------+
only showing top 10 rows


In [38]:
from pyspark.sql.functions import (
    avg,
    sum as spark_sum,
    pow,
    when
)

frequency_means = weekly_purchase_counts.groupBy(
    "user_id"
).agg(
    avg("week_number").alias("mean_week"),
    avg("weekly_purchase_count").alias("mean_weekly_purchases"),
    count("*").alias("observed_weeks")
)

frequency_with_means = weekly_purchase_counts.join(
    frequency_means,
    on="user_id",
    how="left"
)

frequency_slope_components = frequency_with_means.groupBy(
    "user_id"
).agg(
    spark_sum(
        (col("week_number") - col("mean_week")) *
        (col("weekly_purchase_count") - col("mean_weekly_purchases"))
    ).alias("numerator"),

    spark_sum(
        pow(
            col("week_number") - col("mean_week"),
            2
        )
    ).alias("denominator"),

    spark_max("observed_weeks").alias("observed_weeks")
)

frequency_slope_df = frequency_slope_components.withColumn(
    "frequency_slope",
    when(
        col("observed_weeks") >= 2,
        col("numerator") / col("denominator")
    ).otherwise(0.0)
).select(
    "user_id",
    "frequency_slope"
)

frequency_slope_df.show(10, truncate=False)

+---------+---------------+
|user_id  |frequency_slope|
+---------+---------------+
|264649825|0.0            |
|303160429|0.0            |
|340041246|1.0            |
|387300134|0.0            |
|396222093|0.0            |
|403013066|0.0            |
|413782099|0.0            |
|415153398|0.0            |
|418115333|0.0            |
|419697235|0.0            |
+---------+---------------+
only showing top 10 rows


In [39]:
from pyspark.sql.functions import (
    lit,
    explode,
    array,
    coalesce,
    avg,
    sum as spark_sum,
    pow,
    when,
    count,
    max as spark_max
)

# Get all purchasing customers
purchasing_customers = session_purchase_df.select(
    "user_id"
).distinct()

# Create the 5 October weeks
weeks_df = spark.range(1, 6).select(
    col("id").cast("int").alias("week_number")
)

# Create every customer × every week combination
customer_weeks = purchasing_customers.crossJoin(weeks_df)

# Add zero for weeks where the customer made no purchase
complete_weekly_counts = customer_weeks.join(
    weekly_purchase_counts,
    on=["user_id", "week_number"],
    how="left"
).withColumn(
    "weekly_purchase_count",
    coalesce(col("weekly_purchase_count"), lit(0))
)

complete_weekly_counts.show(15)

+---------+-----------+---------------------+
|  user_id|week_number|weekly_purchase_count|
+---------+-----------+---------------------+
|514127317|          1|                    2|
|514127317|          5|                    0|
|515977249|          1|                    3|
|515977249|          5|                    1|
|517017904|          2|                    0|
|517017904|          4|                    0|
|518619616|          3|                    0|
|518619616|          5|                    0|
|537632069|          1|                    3|
|541772342|          2|                    0|
|541772342|          3|                    0|
|541772342|          4|                    0|
|546839974|          2|                    2|
|546839974|          3|                    2|
|550221783|          2|                    0|
+---------+-----------+---------------------+
only showing top 15 rows


In [40]:
frequency_means = complete_weekly_counts.groupBy(
    "user_id"
).agg(
    avg("week_number").alias("mean_week"),
    avg("weekly_purchase_count").alias("mean_weekly_purchases")
)

frequency_with_means = complete_weekly_counts.join(
    frequency_means,
    on="user_id",
    how="left"
)

frequency_slope_components = frequency_with_means.groupBy(
    "user_id"
).agg(
    spark_sum(
        (col("week_number") - col("mean_week")) *
        (col("weekly_purchase_count") - col("mean_weekly_purchases"))
    ).alias("numerator"),

    spark_sum(
        pow(
            col("week_number") - col("mean_week"),
            2
        )
    ).alias("denominator")
)

frequency_slope_df = frequency_slope_components.withColumn(
    "frequency_slope",
    when(
        col("denominator") != 0,
        col("numerator") / col("denominator")
    ).otherwise(0.0)
).select(
    "user_id",
    "frequency_slope"
)

frequency_slope_df.show(10, truncate=False)

+---------+----------------------+
|user_id  |frequency_slope       |
+---------+----------------------+
|264649825|-0.4                  |
|303160429|-0.1                  |
|340041246|-0.4                  |
|387300134|0.1                   |
|396222093|0.1                   |
|403013066|-5.551115123125783E-18|
|413782099|-0.20000000000000004  |
|415153398|-0.1                  |
|418115333|0.1                   |
|419697235|0.1                   |
+---------+----------------------+
only showing top 10 rows


In [41]:
frequency_slope_df.select(
    count("*").alias("customers"),
    spark_sum(
        col("frequency_slope").isNull().cast("int")
    ).alias("null_slopes"),
    spark_sum(
        isnan("frequency_slope").cast("int")
    ).alias("nan_slopes"),
    spark_min("frequency_slope").alias("min_slope"),
    spark_max("frequency_slope").alias("max_slope")
).show(truncate=False)

+---------+-----------+----------+---------+---------+
|customers|null_slopes|nan_slopes|min_slope|max_slope|
+---------+-----------+----------+---------+---------+
|347118   |0          |0         |-9.6     |5.9      |
+---------+-----------+----------+---------+---------+



In [42]:
from pyspark.sql.functions import col

customer_features = (
    customer_rfm
    .join(
        amount_slope_df,
        on="user_id",
        how="left"
    )
    .join(
        frequency_slope_df,
        on="user_id",
        how="left"
    )
)

customer_features = customer_features.select(
    "user_id",
    "recency",
    "frequency",
    "monetary",
    "avg_transaction_value",
    "avg_basket_size",
    "amount_slope",
    "frequency_slope"
)

customer_features.show(10, truncate=False)

+---------+-------+---------+------------------+---------------------+------------------+-------------------+--------------------+
|user_id  |recency|frequency|monetary          |avg_transaction_value|avg_basket_size   |amount_slope       |frequency_slope     |
+---------+-------+---------+------------------+---------------------+------------------+-------------------+--------------------+
|512946423|11     |6        |5272.0            |878.6666666666666    |1.0               |-14.776571428571437|-0.7                |
|514440503|31     |1        |180.16            |180.16               |1.0               |0.0                |-0.2                |
|514734316|31     |1        |64.32             |64.32                |1.0               |0.0                |-0.2                |
|515494704|22     |4        |2717.7            |679.425              |1.75              |38.279999999999994 |-0.7000000000000001 |
|518550701|6      |6        |4586.879999999999 |764.4799999999999    |1.33333333333

In [43]:
print("Customers:", customer_features.count())
print("Features:", len(customer_features.columns))

customer_features.printSchema()

Customers: 347118
Features: 8
root
 |-- user_id: integer (nullable = true)
 |-- recency: integer (nullable = true)
 |-- frequency: long (nullable = false)
 |-- monetary: double (nullable = true)
 |-- avg_transaction_value: double (nullable = true)
 |-- avg_basket_size: double (nullable = true)
 |-- amount_slope: double (nullable = true)
 |-- frequency_slope: double (nullable = true)



In [44]:
from pyspark.sql.functions import sum as spark_sum

customer_features.select([
    spark_sum(col(c).isNull().cast("int")).alias(c)
    for c in customer_features.columns
]).show()

+-------+-------+---------+--------+---------------------+---------------+------------+---------------+
|user_id|recency|frequency|monetary|avg_transaction_value|avg_basket_size|amount_slope|frequency_slope|
+-------+-------+---------+--------+---------------------+---------------+------------+---------------+
|      0|      0|        0|       0|                    0|              0|           0|              0|
+-------+-------+---------+--------+---------------------+---------------+------------+---------------+



In [45]:
from pyspark.sql.functions import (
    min as spark_min,
    max as spark_max,
    avg,
    stddev
)

feature_columns = [
    "recency",
    "frequency",
    "monetary",
    "avg_transaction_value",
    "avg_basket_size",
    "amount_slope",
    "frequency_slope"
]

customer_features.select([
    spark_min(c).alias(f"{c}_min")
    for c in feature_columns
] + [
    spark_max(c).alias(f"{c}_max")
    for c in feature_columns
] + [
    avg(c).alias(f"{c}_mean")
    for c in feature_columns
]).show(truncate=False)

+-----------+-------------+------------+-------------------------+-------------------+----------------+-------------------+-----------+-------------+-----------------+-------------------------+-------------------+-----------------+-------------------+------------------+-----------------+-----------------+--------------------------+--------------------+-----------------+--------------------+
|recency_min|frequency_min|monetary_min|avg_transaction_value_min|avg_basket_size_min|amount_slope_min|frequency_slope_min|recency_max|frequency_max|monetary_max     |avg_transaction_value_max|avg_basket_size_max|amount_slope_max |frequency_slope_max|recency_mean      |frequency_mean   |monetary_mean    |avg_transaction_value_mean|avg_basket_size_mean|amount_slope_mean|frequency_slope_mean|
+-----------+-------------+------------+-------------------------+-------------------+----------------+-------------------+-----------+-------------+-----------------+-------------------------+-------------------

In [46]:
from pyspark.sql.functions import skewness

customer_features.select([
    skewness(col(c)).alias(c)
    for c in feature_columns
]).show(truncate=False)

+-------------------+------------------+------------------+---------------------+-----------------+-------------------+------------------+
|recency            |frequency         |monetary          |avg_transaction_value|avg_basket_size  |amount_slope       |frequency_slope   |
+-------------------+------------------+------------------+---------------------+-----------------+-------------------+------------------+
|0.16674223367475058|11.303364844616842|34.076248229795326|3.4961777314458202   |5.596741976548817|-2.6259312811620825|-2.728020717752717|
+-------------------+------------------+------------------+---------------------+-----------------+-------------------+------------------+



In [47]:
customer_features.select([
    spark_min(c).alias(f"{c}_min")
    for c in feature_columns
] + [
    spark_max(c).alias(f"{c}_max")
    for c in feature_columns
] + [
    avg(c).alias(f"{c}_mean")
    for c in feature_columns
] + [
    skewness(c).alias(f"{c}_skewness")
    for c in feature_columns
]).show(truncate=False)

+-----------+-------------+------------+-------------------------+-------------------+----------------+-------------------+-----------+-------------+-----------------+-------------------------+-------------------+-----------------+-------------------+------------------+-----------------+-----------------+--------------------------+--------------------+-----------------+--------------------+-------------------+------------------+------------------+------------------------------+------------------------+---------------------+------------------------+
|recency_min|frequency_min|monetary_min|avg_transaction_value_min|avg_basket_size_min|amount_slope_min|frequency_slope_min|recency_max|frequency_max|monetary_max     |avg_transaction_value_max|avg_basket_size_max|amount_slope_max |frequency_slope_max|recency_mean      |frequency_mean   |monetary_mean    |avg_transaction_value_mean|avg_basket_size_mean|amount_slope_mean|frequency_slope_mean|recency_skewness   |frequency_skewness|monetary_skew

In [48]:
from pyspark.sql.functions import (
    min as spark_min,
    max as spark_max,
    avg,
    skewness
)

stats = []

for c in feature_columns:
    row = customer_features.select(
        spark_min(c).alias("min"),
        spark_max(c).alias("max"),
        avg(c).alias("mean"),
        skewness(c).alias("skewness")
    ).collect()[0]

    stats.append((
        c,
        row["min"],
        row["max"],
        row["mean"],
        row["skewness"]
    ))

for feature, min_val, max_val, mean_val, skew_val in stats:
    print(f"{feature}")
    print(f"  Min       : {min_val}")
    print(f"  Max       : {max_val}")
    print(f"  Mean      : {mean_val}")
    print(f"  Skewness  : {skew_val}")
    print()

recency
  Min       : 1
  Max       : 31
  Mean      : 14.599565565600171
  Skewness  : 0.166742233674755

frequency
  Min       : 1
  Max       : 152
  Mean      : 1.813677193346355
  Skewness  : 11.303364844617008

monetary
  Min       : 0.88
  Max       : 265569.5199999999
  Mean      : 662.4764554704582
  Skewness  : 34.07624822979402

avg_transaction_value
  Min       : 0.88
  Max       : 9356.73
  Mean      : 317.4216344376997
  Skewness  : 3.4961777314458917

avg_basket_size
  Min       : 1.0
  Max       : 20.0
  Mean      : 1.1280295687752255
  Skewness  : 5.596741976548596

amount_slope
  Min       : -10476.18
  Max       : 8751.469999999998
  Mean      : -9.25052150338933
  Skewness  : -2.6259312811620825

frequency_slope
  Min       : -9.6
  Max       : 5.9
  Mean      : -0.04682845602935486
  Skewness  : -2.728020717752717



## FEATURE TRANSFORMATION


In [49]:
from pyspark.sql.functions import log1p, abs, signum

transformed_features = customer_features

# Log transformation for positively skewed features
transformed_features = (
    transformed_features
    .withColumn("frequency_transformed", log1p(col("frequency")))
    .withColumn("monetary_transformed", log1p(col("monetary")))
    .withColumn(
        "avg_transaction_value_transformed",
        log1p(col("avg_transaction_value"))
    )
    .withColumn(
        "avg_basket_size_transformed",
        log1p(col("avg_basket_size"))
    )
)

# Signed log transformation for slope features
transformed_features = (
    transformed_features
    .withColumn(
        "amount_slope_transformed",
        signum(col("amount_slope")) * log1p(abs(col("amount_slope")))
    )
    .withColumn(
        "frequency_slope_transformed",
        signum(col("frequency_slope")) * log1p(abs(col("frequency_slope")))
    )
)

# Recency is retained without transformation
transformed_features = transformed_features.withColumn(
    "recency_transformed",
    col("recency")
)

transformed_features.select(
    "user_id",
    "recency_transformed",
    "frequency_transformed",
    "monetary_transformed",
    "avg_transaction_value_transformed",
    "avg_basket_size_transformed",
    "amount_slope_transformed",
    "frequency_slope_transformed"
).show(10, truncate=False)

+---------+-------------------+---------------------+--------------------+---------------------------------+---------------------------+------------------------+---------------------------+
|user_id  |recency_transformed|frequency_transformed|monetary_transformed|avg_transaction_value_transformed|avg_basket_size_transformed|amount_slope_transformed|frequency_slope_transformed|
+---------+-------------------+---------------------+--------------------+---------------------------------+---------------------------+------------------------+---------------------------+
|512946423|11                 |1.9459101490553132   |8.570354739530469   |6.779543047835214                |0.6931471805599453         |-2.758526018589407      |-0.5306282510621704        |
|514440503|31                 |0.6931471805599453   |5.199380618688013   |5.199380618688013                |0.6931471805599453         |0.0                     |-0.18232155679395462       |
|514734316|31                 |0.6931471805599453 

In [50]:
transformed_columns = [
    "recency_transformed",
    "frequency_transformed",
    "monetary_transformed",
    "avg_transaction_value_transformed",
    "avg_basket_size_transformed",
    "amount_slope_transformed",
    "frequency_slope_transformed"
]

transformed_features.select([
    skewness(col(c)).alias(c)
    for c in transformed_columns
]).show(truncate=False)

+-------------------+---------------------+--------------------+---------------------------------+---------------------------+------------------------+---------------------------+
|recency_transformed|frequency_transformed|monetary_transformed|avg_transaction_value_transformed|avg_basket_size_transformed|amount_slope_transformed|frequency_slope_transformed|
+-------------------+---------------------+--------------------+---------------------------------+---------------------------+------------------------+---------------------------+
|0.16674223367475058|2.4837223330464453   |0.11686747221025964 |-0.22215172015214235             |3.419363569080542          |-0.1819474018749545     |-0.5621682027698194        |
+-------------------+---------------------+--------------------+---------------------------------+---------------------------+------------------------+---------------------------+



In [51]:
from pyspark.ml.feature import VectorAssembler

transformed_feature_columns = [
    "recency_transformed",
    "frequency_transformed",
    "monetary_transformed",
    "avg_transaction_value_transformed",
    "avg_basket_size_transformed",
    "amount_slope_transformed",
    "frequency_slope_transformed"
]

assembler = VectorAssembler(
    inputCols=transformed_feature_columns,
    outputCol="features_raw"
)

assembled_features = assembler.transform(transformed_features)

assembled_features.select(
    "user_id",
    "features_raw"
).show(10, truncate=False)

+---------+-------------------------------------------------------------------------------------------------------------------------+
|user_id  |features_raw                                                                                                             |
+---------+-------------------------------------------------------------------------------------------------------------------------+
|264649825|[26.0,1.0986122886681096,7.123705016756076,6.431363287534036,0.6931471805599453,4.918227564771101,-0.33647223662121295]  |
|303160429|[19.0,0.6931471805599453,5.833611187481239,5.833611187481239,0.6931471805599453,0.0,-0.09531017980432487]                |
|340041246|[18.0,1.3862943611198906,6.820583889191142,5.72415139038256,0.8472978603872036,-2.933856869835904,-0.33647223662121295]  |
|387300134|[6.0,0.6931471805599453,3.072230244526716,3.072230244526716,0.6931471805599453,0.0,0.09531017980432487]                  |
|396222093|[8.0,0.6931471805599453,3.894673367093033,3.8946733

In [52]:
from pyspark.ml.feature import StandardScaler

scaler = StandardScaler(
    inputCol="features_raw",
    outputCol="features",
    withStd=True,
    withMean=True
)

scaler_model = scaler.fit(assembled_features)

scaled_features = scaler_model.transform(assembled_features)

scaled_features.select(
    "user_id",
    "features"
).show(10, truncate=False)

+---------+--------------------------------------------------------------------------------------------------------------------------------------------+
|user_id  |features                                                                                                                                    |
+---------+--------------------------------------------------------------------------------------------------------------------------------------------+
|264649825|[1.3148241067808233,0.45793841497915355,1.1910884731558815,1.108083100996223,-0.3637197122113236,2.2544213175288483,-1.5765650189431226]    |
|303160429|[0.5075067365151686,-0.5488148614605252,0.22801437663656815,0.575040295444265,-0.3637197122113236,0.07441491538210748,-0.3018023526423277]  |
|340041246|[0.3921756836200751,1.1722412322181786,0.964804074733786,0.4774300029242149,0.7447540630430974,-1.2260183555460824,-1.5765650189431226]     |
|387300134|[-0.991796951121047,-0.5488148614605252,-1.833397279193486,-1.887409012

In [54]:
# Sample customers for K selection
kmeans_sample = scaled_features.sample(
    withReplacement=False,
    fraction=0.15,
    seed=42
).cache()

print("Sample customers:", kmeans_sample.count())

Sample customers: 52518


In [55]:
from pyspark.ml.clustering import KMeans

k_values = list(range(2, 11))
wss_values = []

for k in k_values:
    print(f"Running K = {k}...")

    kmeans = KMeans(
        featuresCol="features",
        predictionCol="prediction",
        k=k,
        seed=42,
        maxIter=20
    )

    model = kmeans.fit(kmeans_sample)
    wss_values.append(model.summary.trainingCost)

print("\nElbow Results:")
for k, wss in zip(k_values, wss_values):
    print(f"K = {k}, WCSS = {wss}")

Running K = 2...
Running K = 3...
Running K = 4...
Running K = 5...
Running K = 6...
Running K = 7...
Running K = 8...
Running K = 9...
Running K = 10...

Elbow Results:
K = 2, WCSS = 290953.41607384617
K = 3, WCSS = 253628.01605503607
K = 4, WCSS = 222139.58237767487
K = 5, WCSS = 185547.83152056183
K = 6, WCSS = 165757.23109256936
K = 7, WCSS = 150893.6054244773
K = 8, WCSS = 143277.63447274323
K = 9, WCSS = 126273.24560928947
K = 10, WCSS = 119493.8626684824
